# LiDAR Obstacle Detection - PointNet Training
**Airbus AI Hackathon 2026**

Ce notebook permet d'entraîner le modèle PointNet sur GPU Google Colab.

## Instructions
1. **Runtime > Change runtime type > GPU (T4)**
2. Exécutez les cellules dans l'ordre
3. Uploadez vos fichiers HDF5 dans Google Drive

## 1. Vérification GPU et Installation

In [ ]:
# Vérifier le GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Installer les dépendances
!pip install h5py pandas tqdm -q

## 2. Monter Google Drive et Configurer les Données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Créer les dossiers nécessaires
!mkdir -p /content/drive/MyDrive/LiDAR_Hackathon/data
!mkdir -p /content/drive/MyDrive/LiDAR_Hackathon/checkpoints

print("\n" + "="*50)
print("IMPORTANT: Uploadez vos fichiers scene_*.h5 dans:")
print("/content/drive/MyDrive/LiDAR_Hackathon/data/")
print("="*50)

In [ ]:
# Vérifier les fichiers de données
import os

DATA_DIR = "/content/drive/MyDrive/LiDAR_Hackathon/data"
CHECKPOINT_DIR = "/content/drive/MyDrive/LiDAR_Hackathon/checkpoints"

h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5')]
print(f"Fichiers HDF5 trouvés: {len(h5_files)}")
for f in sorted(h5_files):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
    print(f"  - {f} ({size_mb:.1f} MB)")

## 3. Code Source

In [ ]:
# ============================================
# lidar_utils.py
# ============================================

import h5py
import numpy as np
import pandas as pd

def load_h5_data(file_path, dataset_name="lidar_points"):
    """Loads HDF5 data into a Pandas DataFrame."""
    with h5py.File(file_path, "r") as f:
        if dataset_name not in f:
            raise ValueError(f"Dataset '{dataset_name}' not found in {file_path}")
        points = f[dataset_name][:]
    return pd.DataFrame({name: points[name] for name in points.dtype.names})

def get_unique_poses(df):
    """Returns a DataFrame of unique poses with a 'pose_index'."""
    pose_fields = ["ego_x", "ego_y", "ego_z", "ego_yaw"]
    if not all(f in df.columns for f in pose_fields):
        return None
    pose_counts = (
        df.groupby(pose_fields)
        .size()
        .reset_index(name="num_points")
        .reset_index(names="pose_index")
    )
    return pose_counts

def filter_by_pose(df, pose_row):
    """Filters the dataframe for a specific pose quadruplet."""
    return df[
        (df["ego_x"] == pose_row["ego_x"]) &
        (df["ego_y"] == pose_row["ego_y"]) &
        (df["ego_z"] == pose_row["ego_z"]) &
        (df["ego_yaw"] == pose_row["ego_yaw"])
    ].reset_index(drop=True)

def spherical_to_local_cartesian(df):
    """Converts spherical coordinates to local Cartesian (Lidar Frame)."""
    distance_m = df["distance_cm"].to_numpy() / 100.0
    azimuth_rad = np.radians(df["azimuth_raw"] / 100.0)
    elevation_rad = np.radians(df["elevation_raw"] / 100.0)

    x = distance_m * np.cos(elevation_rad) * np.cos(azimuth_rad)
    y = -distance_m * np.cos(elevation_rad) * np.sin(azimuth_rad)
    z = distance_m * np.sin(elevation_rad)

    return np.column_stack((x, y, z))

print("lidar_utils loaded")

In [ ]:
# ============================================
# dataset.py
# ============================================

from torch.utils.data import Dataset

# Class color mapping (RGB values)
CLASS_COLORS = {
    0: (38, 23, 180),      # Antenna
    1: (177, 132, 47),     # Cable
    2: (129, 81, 97),      # Electric pole
    3: (66, 132, 9),       # Wind turbine
}

CLASS_NAMES = {
    0: "Antenna",
    1: "Cable",
    2: "Electric pole",
    3: "Wind turbine",
    4: "Background"
}

NUM_CLASSES = 5  # 4 obstacles + 1 background


def rgb_to_class_id(r, g, b):
    """Map RGB tuple to class ID."""
    for class_id, (cr, cg, cb) in CLASS_COLORS.items():
        if r == cr and g == cg and b == cb:
            return class_id
    return 4  # Background


def rgb_array_to_class_ids(rgb_array):
    """Convert array of RGB values to class IDs."""
    r, g, b = rgb_array[:, 0], rgb_array[:, 1], rgb_array[:, 2]
    class_ids = np.full(len(rgb_array), 4, dtype=np.int64)  # Default: background

    for class_id, (cr, cg, cb) in CLASS_COLORS.items():
        mask = (r == cr) & (g == cg) & (b == cb)
        class_ids[mask] = class_id

    return class_ids


class LidarDataset(Dataset):
    """PyTorch Dataset for LiDAR point cloud segmentation."""

    def __init__(self, data_dir, n_points=32768, augment=True, normalize=True, use_reflectivity=True):
        self.data_dir = data_dir
        self.n_points = n_points
        self.augment = augment
        self.normalize = normalize
        self.use_reflectivity = use_reflectivity
        self.frames = []
        self._load_all_frames()

    def _load_all_frames(self):
        """Load metadata for all frames from all HDF5 files."""
        h5_files = [f for f in os.listdir(self.data_dir) if f.endswith('.h5')]

        for h5_file in sorted(h5_files):
            file_path = os.path.join(self.data_dir, h5_file)
            try:
                df = load_h5_data(file_path)
                pose_counts = get_unique_poses(df)

                if pose_counts is not None:
                    for idx in range(len(pose_counts)):
                        self.frames.append({
                            'file_path': file_path,
                            'pose_index': idx,
                            'pose_data': pose_counts.iloc[idx].to_dict()
                        })
            except Exception as e:
                print(f"Warning: Could not load {h5_file}: {e}")

        print(f"Loaded {len(self.frames)} frames from {len(h5_files)} files")

    def __len__(self):
        return len(self.frames)

    def _augment_points(self, xyz):
        """Apply data augmentation to point cloud."""
        theta = np.random.uniform(0, 2 * np.pi)
        rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta), np.cos(theta), 0],
            [0, 0, 1]
        ])
        xyz = xyz @ rotation_matrix.T
        jitter = np.random.normal(0, 0.01, size=xyz.shape)
        xyz = xyz + jitter
        scale = np.random.uniform(0.9, 1.1)
        xyz = xyz * scale
        return xyz

    def _normalize_points(self, xyz):
        """Normalize point cloud to unit sphere."""
        centroid = np.mean(xyz, axis=0)
        xyz = xyz - centroid
        max_dist = np.max(np.sqrt(np.sum(xyz ** 2, axis=1)))
        if max_dist > 0:
            xyz = xyz / max_dist
        return xyz

    def _sample_points(self, xyz, features, labels):
        """Sample or pad point cloud to fixed size."""
        n_points_current = xyz.shape[0]

        if n_points_current == 0:
            xyz = np.zeros((self.n_points, 3), dtype=np.float32)
            features = np.zeros(self.n_points, dtype=np.float32) if features is not None else None
            labels = np.full(self.n_points, 4, dtype=np.int64)
            return xyz, features, labels

        if n_points_current >= self.n_points:
            indices = np.random.choice(n_points_current, self.n_points, replace=False)
        else:
            indices = np.random.choice(n_points_current, self.n_points, replace=True)

        xyz = xyz[indices]
        features = features[indices] if features is not None else None
        labels = labels[indices]

        return xyz, features, labels

    def __getitem__(self, idx):
        frame_info = self.frames[idx]

        df = load_h5_data(frame_info['file_path'])
        pose_counts = get_unique_poses(df)
        selected_pose = pose_counts.iloc[frame_info['pose_index']]
        frame_df = filter_by_pose(df, selected_pose)

        valid_mask = frame_df['distance_cm'] > 0
        frame_df = frame_df[valid_mask].reset_index(drop=True)

        xyz = spherical_to_local_cartesian(frame_df)

        rgb_values = frame_df[['r', 'g', 'b']].values
        labels = rgb_array_to_class_ids(rgb_values)

        reflectivity = None
        if self.use_reflectivity and 'reflectivity' in frame_df.columns:
            reflectivity = frame_df['reflectivity'].values.astype(np.float32) / 255.0

        xyz, reflectivity, labels = self._sample_points(xyz, reflectivity, labels)

        if self.normalize:
            xyz = self._normalize_points(xyz)

        if self.augment:
            xyz = self._augment_points(xyz)

        xyz = xyz.astype(np.float32)
        if self.use_reflectivity and reflectivity is not None:
            features = np.column_stack([xyz, reflectivity])
        else:
            features = xyz

        features = torch.from_numpy(features).float()
        labels = torch.from_numpy(labels).long()

        return features, labels

    def get_class_weights(self):
        """Compute class weights based on inverse frequency."""
        class_counts = np.zeros(NUM_CLASSES, dtype=np.float64)

        sample_size = min(20, len(self.frames))
        sample_indices = np.random.choice(len(self.frames), sample_size, replace=False)

        for idx in sample_indices:
            _, labels = self[idx]
            for c in range(NUM_CLASSES):
                class_counts[c] += (labels == c).sum().item()

        total = class_counts.sum()
        weights = total / (NUM_CLASSES * class_counts + 1e-6)
        weights = weights / weights.sum() * NUM_CLASSES

        return torch.from_numpy(weights).float()


def get_dataloaders(data_dir, batch_size=8, n_points=32768, num_workers=2, val_split=0.2):
    """Create train and validation dataloaders."""
    full_dataset = LidarDataset(
        data_dir=data_dir,
        n_points=n_points,
        augment=True,
        normalize=True,
        use_reflectivity=True
    )

    n_val = int(len(full_dataset) * val_split)
    n_train = len(full_dataset) - n_val

    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )

    pin_memory = torch.cuda.is_available()

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, full_dataset.get_class_weights()

print("dataset loaded")

In [ ]:
# ============================================
# pointnet_model.py
# ============================================

import torch.nn as nn
import torch.nn.functional as F


class TNet(nn.Module):
    """Transformation Network for learning spatial/feature transformations."""

    def __init__(self, k=3):
        super().__init__()
        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)

        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k * k)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)

    def forward(self, x):
        batch_size = x.size(0)

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))

        x = torch.max(x, 2)[0]

        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)

        identity = torch.eye(self.k, device=x.device, dtype=x.dtype)
        identity = identity.view(1, self.k * self.k).repeat(batch_size, 1)

        x = x + identity
        x = x.view(-1, self.k, self.k)

        return x


class PointNetEncoder(nn.Module):
    """PointNet encoder with T-Net transformations."""

    def __init__(self, input_channels=4, feature_transform=True):
        super().__init__()
        self.feature_transform = feature_transform

        self.tnet3 = TNet(k=3)

        self.conv1 = nn.Conv1d(input_channels, 64, 1)
        self.conv2 = nn.Conv1d(64, 64, 1)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)

        if feature_transform:
            self.tnet64 = TNet(k=64)

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.conv4 = nn.Conv1d(64, 128, 1)
        self.conv5 = nn.Conv1d(128, 1024, 1)

        self.bn3 = nn.BatchNorm1d(64)
        self.bn4 = nn.BatchNorm1d(128)
        self.bn5 = nn.BatchNorm1d(1024)

    def forward(self, x):
        batch_size, n_points, _ = x.size()

        xyz = x[:, :, :3]

        xyz_t = xyz.transpose(2, 1)
        trans3 = self.tnet3(xyz_t)
        xyz = torch.bmm(xyz, trans3)

        if x.size(2) > 3:
            x = torch.cat([xyz, x[:, :, 3:]], dim=2)
        else:
            x = xyz

        x = x.transpose(2, 1)

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        trans64 = None
        if self.feature_transform:
            trans64 = self.tnet64(x)
            x = x.transpose(2, 1)
            x = torch.bmm(x, trans64)
            x = x.transpose(2, 1)

        point_features = x

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))

        global_feature = torch.max(x, 2)[0]

        return global_feature, point_features, trans3, trans64


class PointNetSegmentation(nn.Module):
    """PointNet for semantic segmentation."""

    def __init__(self, num_classes=5, input_channels=4, feature_transform=True):
        super().__init__()
        self.num_classes = num_classes
        self.feature_transform = feature_transform

        self.encoder = PointNetEncoder(
            input_channels=input_channels,
            feature_transform=feature_transform
        )

        self.conv1 = nn.Conv1d(1088, 512, 1)
        self.conv2 = nn.Conv1d(512, 256, 1)
        self.conv3 = nn.Conv1d(256, 128, 1)
        self.conv4 = nn.Conv1d(128, num_classes, 1)

        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        batch_size, n_points, _ = x.size()

        global_feature, point_features, trans3, trans64 = self.encoder(x)

        global_feature = global_feature.unsqueeze(2).repeat(1, 1, n_points)
        concat_features = torch.cat([point_features, global_feature], dim=1)

        x = F.relu(self.bn1(self.conv1(concat_features)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(x)
        x = self.conv4(x)

        x = x.transpose(2, 1)

        return x, trans3, trans64


def feature_transform_regularizer(trans):
    """Regularization loss for feature transformation matrix."""
    if trans is None:
        return 0

    d = trans.size(1)
    batch_size = trans.size(0)

    identity = torch.eye(d, device=trans.device, dtype=trans.dtype)
    identity = identity.unsqueeze(0).repeat(batch_size, 1, 1)

    loss = torch.mean(torch.norm(torch.bmm(trans, trans.transpose(2, 1)) - identity, dim=(1, 2)))
    return loss


def get_model_size(model):
    """Return the number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("pointnet_model loaded")

## 4. Entraînement

In [ ]:
# ============================================
# Fonctions d'entraînement
# ============================================

import time
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR


def compute_iou(pred, target, num_classes):
    """Compute IoU per class."""
    ious = []
    pred = pred.reshape(-1)
    target = target.reshape(-1)

    for cls in range(num_classes):
        pred_cls = pred == cls
        target_cls = target == cls

        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()

        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append((intersection / union).item())

    return ious


def train_one_epoch(model, train_loader, criterion, optimizer, device, feature_reg_weight=0.001):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_correct = 0
    total_points = 0

    pbar = tqdm(train_loader, desc="Training")
    for features, labels in pbar:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs, trans3, trans64 = model(features)
        outputs = outputs.reshape(-1, NUM_CLASSES)
        labels = labels.reshape(-1)

        loss = criterion(outputs, labels)

        reg_loss = feature_transform_regularizer(trans64)
        loss = loss + feature_reg_weight * reg_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = outputs.argmax(dim=1)
        total_correct += (pred == labels).sum().item()
        total_points += labels.numel()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * total_correct / total_points:.2f}%'
        })

    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_points

    return avg_loss, accuracy


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    total_correct = 0
    total_points = 0
    all_ious = []

    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc="Validation"):
            features = features.to(device)
            labels = labels.to(device)

            outputs, _, _ = model(features)
            outputs_flat = outputs.reshape(-1, NUM_CLASSES)
            labels_flat = labels.reshape(-1)

            loss = criterion(outputs_flat, labels_flat)
            total_loss += loss.item()

            pred = outputs_flat.argmax(dim=1)
            total_correct += (pred == labels_flat).sum().item()
            total_points += labels_flat.numel()

            pred_batch = outputs.argmax(dim=2)
            batch_ious = compute_iou(pred_batch, labels, NUM_CLASSES)
            all_ious.append(batch_ious)

    avg_loss = total_loss / len(val_loader)
    accuracy = total_correct / total_points

    all_ious = np.array(all_ious)
    mean_ious = np.nanmean(all_ious, axis=0)
    miou = np.nanmean(mean_ious[:4])  # mIoU for obstacle classes only

    return avg_loss, accuracy, mean_ious, miou

print("training functions loaded")

In [ ]:
# ============================================
# Configuration de l'entraînement
# ============================================

# Hyperparamètres
EPOCHS = 100
BATCH_SIZE = 8
LEARNING_RATE = 0.001
N_POINTS = 32768
NUM_WORKERS = 2

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Charger les données
print("\nLoading data...")
train_loader, val_loader, class_weights = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    n_points=N_POINTS,
    num_workers=NUM_WORKERS
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Class weights: {class_weights}")

# Créer le modèle
model = PointNetSegmentation(num_classes=NUM_CLASSES, input_channels=4)
model = model.to(device)
print(f"\nModel parameters: {get_model_size(model):,}")

# Loss, optimizer, scheduler
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [ ]:
# ============================================
# Boucle d'entraînement
# ============================================

best_miou = 0

print("\n" + "="*50)
print("Starting training...")
print("="*50)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    start_time = time.time()

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_loss, val_acc, class_ious, miou = validate(
        model, val_loader, criterion, device
    )

    # Update scheduler
    scheduler.step()

    epoch_time = time.time() - start_time

    # Print metrics
    print(f"Time: {epoch_time:.1f}s | LR: {scheduler.get_last_lr()[0]:.6f}")
    print(f"Train - Loss: {train_loss:.4f}, Acc: {100*train_acc:.2f}%")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {100*val_acc:.2f}%, mIoU: {100*miou:.2f}%")
    print("IoU per class:")
    for c in range(NUM_CLASSES):
        iou = class_ious[c]
        if not np.isnan(iou):
            print(f"  {CLASS_NAMES[c]}: {100*iou:.2f}%")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'miou': miou,
        'best_miou': best_miou
    }

    # Save latest
    torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'latest.pth'))

    # Save best
    if miou > best_miou:
        best_miou = miou
        checkpoint['best_miou'] = best_miou
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'best.pth'))
        print(f">>> New best mIoU: {100*best_miou:.2f}% <<<")

    # Save periodic checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1}.pth'))

print(f"\n" + "="*50)
print(f"Training complete! Best mIoU: {100*best_miou:.2f}%")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print("="*50)

## 5. Télécharger le modèle entraîné

In [ ]:
# Télécharger le meilleur modèle
from google.colab import files

best_model_path = os.path.join(CHECKPOINT_DIR, 'best.pth')
if os.path.exists(best_model_path):
    files.download(best_model_path)
    print(f"Downloaded: {best_model_path}")
else:
    print("No best.pth found. Training may not have completed.")

## 6. Exporter en ONNX (optionnel)

In [ ]:
# Exporter le modèle en ONNX
import torch.onnx

# Charger le meilleur modèle
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Créer un input factice
dummy_input = torch.randn(1, N_POINTS, 4).to(device)

# Exporter
onnx_path = os.path.join(CHECKPOINT_DIR, 'pointnet_segmentation.onnx')
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['point_cloud'],
    output_names=['segmentation', 'trans3', 'trans64'],
    dynamic_axes={
        'point_cloud': {0: 'batch_size', 1: 'num_points'},
        'segmentation': {0: 'batch_size', 1: 'num_points'}
    },
    opset_version=11
)

print(f"ONNX model saved to: {onnx_path}")

# Télécharger
files.download(onnx_path)